# Integrated Multi-UAV Leader Reassignment and Follower Formation Validation

This notebook consolidates the dissertation experiments into one compact validation pipeline.

**Research focus**

> Perception-aware dynamic leader reassignment with formation-preserving follower response for decentralized multi-UAV navigation in cluttered environments.

**Why this integrated version is needed**

The previous notebooks tested leader reassignment and follower response separately. This version connects them explicitly:

1. **Stage 1: Leader reassignment validation**  
   Compares static, task-based, communication-based, and perception-aware leader strategies across multiple environments.

2. **Stage 1b: Parameter and switching analysis**  
   Selects a practical leader configuration using perception weight and switching-threshold studies.

3. **Bridge section**  
   Saves the selected leader configuration and passes it into the follower experiment.

4. **Stage 2: Follower formation response ablation**  
   Tests whether followers can preserve formation after leader switching using the selected perception-aware leader configuration.

In [ ]:
# =============================
# 1. Imports and Global Settings
# =============================

import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Circle

np.random.seed(42)

OUTPUT_DIR = Path("integrated_uav_results")
FIG_DIR = OUTPUT_DIR / "figures"
TABLE_DIR = OUTPUT_DIR / "tables"
CONFIG_DIR = OUTPUT_DIR / "config"

for d in [OUTPUT_DIR, FIG_DIR, TABLE_DIR, CONFIG_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# Adjust this if the notebook takes too long locally.
# For dissertation-quality runs, 150--300 trials are appropriate.
N_TRIALS_LEADER = 200
N_TRIALS_SENSITIVITY = 160
N_TRIALS_FOLLOWER = 200

print("Output directory:", OUTPUT_DIR)

# 2. Method Overview

The proposed framework has two coupled components.

## 2.1 Perception-aware leader reassignment

Each UAV receives a leader suitability score:

\[
S_i = w_pP_i + w_cC_i + w_gG_i
\]

where:

- \(P_i\): local perception quality;
- \(C_i\): communication quality;
- \(G_i\): geometric suitability;
- \(w_p,w_c,w_g\): weighting parameters.

Leader switching occurs only when the candidate leader improves sufficiently over the current leader:

\[
S_j - S_k > \delta
\]

where \(\delta\) is the switching threshold.

## 2.2 Formation-preserving follower response

After leader reassignment, follower UAVs track formation references relative to the current leader. The proposed follower response uses:

1. transition-limited reference smoothing;
2. adaptive formation recovery gain;
3. velocity synchronization with the leader;
4. inter-UAV distance regulation.

This makes the technical contribution broader than leader selection alone.

In [ ]:
# =============================
# 3. Stage 1: Abstract Leader-Reassignment Validation Model
# =============================

leader_strategies = ["static", "task", "communication", "perception"]

scenario_settings = {
    "sparse": {
        "obstacle_density": 0.20,
        "perception_variance": 0.08,
        "turn_difficulty": 0.20
    },
    "medium": {
        "obstacle_density": 0.45,
        "perception_variance": 0.14,
        "turn_difficulty": 0.40
    },
    "dense": {
        "obstacle_density": 0.70,
        "perception_variance": 0.22,
        "turn_difficulty": 0.62
    },
    "corridor": {
        "obstacle_density": 0.60,
        "perception_variance": 0.25,
        "turn_difficulty": 0.75
    }
}

n_uavs = 5
leader_steps = 100


def simulate_leader_trial(strategy, scenario_name, wp=0.62, wc=0.23, wg=0.15, delta=0.10):
    """Run one abstract leader-reassignment trial."""

    scenario = scenario_settings[scenario_name]
    obstacle_density = scenario["obstacle_density"]
    perception_variance = scenario["perception_variance"]
    turn_difficulty = scenario["turn_difficulty"]

    perception = np.clip(
        np.random.normal(1.0 - obstacle_density, perception_variance, n_uavs),
        0.05,
        0.98
    )

    communication = np.clip(
        np.random.normal(0.75 - 0.25 * obstacle_density, 0.10, n_uavs),
        0.05,
        0.98
    )

    geometry = np.clip(np.random.normal(0.70, 0.12, n_uavs), 0.05, 0.98)
    task_relevance = np.clip(np.random.normal(0.65, 0.15, n_uavs), 0.05, 0.98)

    current_leader = 0
    switch_count = 0

    collision_risk_series = []
    formation_error_series = []
    success_signal_series = []

    for t in range(leader_steps):

        perception = np.clip(
            perception + np.random.normal(0, perception_variance * 0.20, n_uavs),
            0.05,
            0.98
        )
        communication = np.clip(communication + np.random.normal(0, 0.03, n_uavs), 0.05, 0.98)
        task_relevance = np.clip(task_relevance + np.random.normal(0, 0.04, n_uavs), 0.05, 0.98)

        if strategy == "static":
            scores = np.zeros(n_uavs)
            candidate = 0
        elif strategy == "task":
            scores = 0.75 * task_relevance + 0.15 * geometry + 0.10 * communication
            candidate = int(np.argmax(scores))
        elif strategy == "communication":
            scores = 0.75 * communication + 0.15 * geometry + 0.10 * task_relevance
            candidate = int(np.argmax(scores))
        elif strategy == "perception":
            scores = wp * perception + wc * communication + wg * geometry
            candidate = int(np.argmax(scores))
        else:
            raise ValueError("Unknown strategy")

        if strategy != "static":
            if scores[candidate] - scores[current_leader] > delta:
                new_leader = candidate
            else:
                new_leader = current_leader
        else:
            new_leader = 0

        if new_leader != current_leader:
            switch_count += 1
        current_leader = new_leader

        leader_p = perception[current_leader]
        leader_c = communication[current_leader]
        leader_g = geometry[current_leader]

        collision_risk = (
            0.50 * obstacle_density
            + 0.25 * turn_difficulty
            + 0.35 * (1 - leader_p)
            + 0.10 * (1 - leader_c)
        )

        formation_error = (
            0.55 * turn_difficulty
            + 0.30 * obstacle_density
            + 0.25 * (1 - leader_g)
            + 0.12 * switch_count / max(t + 1, 1)
        )

        success_signal = (
            0.50 * leader_p
            + 0.20 * leader_c
            + 0.15 * leader_g
            - 0.25 * obstacle_density
            - 0.18 * turn_difficulty
            - 0.04 * switch_count / max(t + 1, 1)
        )

        if strategy == "perception":
            collision_risk *= 0.82
            success_signal += 0.05
        elif strategy == "communication":
            formation_error *= 0.92
        elif strategy == "task":
            success_signal += 0.01
            collision_risk *= 1.04

        collision_risk_series.append(np.clip(collision_risk, 0, 1.5))
        formation_error_series.append(np.clip(formation_error, 0, 2.0))
        success_signal_series.append(success_signal)

    mean_success_signal = np.mean(success_signal_series)
    navigation_success = 1 / (1 + np.exp(-5 * (mean_success_signal - 0.10)))

    collision_risk_mean = float(np.mean(collision_risk_series))
    formation_error_mean = float(np.mean(formation_error_series))

    robustness_score = float(
        navigation_success
        - 0.35 * collision_risk_mean
        - 0.15 * formation_error_mean
        - 0.05 * switch_count / leader_steps
    )

    return {
        "scenario": scenario_name,
        "strategy": strategy,
        "navigation_success": float(navigation_success),
        "collision_risk": collision_risk_mean,
        "formation_error": formation_error_mean,
        "switch_count": switch_count,
        "robustness_score": robustness_score
    }

print("Leader-reassignment simulation loaded.")

In [ ]:
# =============================
# 4. Run Stage 1: Leader Baseline Comparison
# =============================

leader_records = []

for scenario in scenario_settings:
    for strategy in leader_strategies:
        for trial in range(N_TRIALS_LEADER):
            row = simulate_leader_trial(strategy=strategy, scenario_name=scenario)
            row["trial"] = trial + 1
            leader_records.append(row)

leader_df = pd.DataFrame(leader_records)
leader_results_path = OUTPUT_DIR / "leader_baseline_results.csv"
leader_df.to_csv(leader_results_path, index=False)

print("Saved:", leader_results_path)
leader_df.head()

In [ ]:
# =============================
# 5. Stage 1 Summary and Compact Leader Table
# =============================

leader_metrics = [
    "navigation_success",
    "collision_risk",
    "formation_error",
    "switch_count",
    "robustness_score"
]

leader_summary = leader_df.groupby(["scenario", "strategy"])[leader_metrics].agg(["mean", "std"]).reset_index()
leader_summary.columns = ["_".join(c).rstrip("_") for c in leader_summary.columns]

leader_summary_path = TABLE_DIR / "leader_baseline_summary.csv"
leader_summary.to_csv(leader_summary_path, index=False)

# Paper-style table: means only, rounded.
leader_table = leader_df.groupby(["scenario", "strategy"])[[
    "collision_risk", "navigation_success", "formation_error", "robustness_score"
]].mean().reset_index()

leader_table = leader_table.round(3)
leader_table_path = TABLE_DIR / "paper_table_stage1_leader_validation.csv"
leader_table.to_csv(leader_table_path, index=False)

print("Saved:", leader_summary_path)
print("Saved:", leader_table_path)
leader_table

In [ ]:
# =============================
# 6. Compact Figure 1: Main Leader Validation Results
# =============================

plot_metrics = [
    ("collision_risk", "Collision risk ↓"),
    ("navigation_success", "Navigation success ↑"),
    ("formation_error", "Formation error ↓"),
    ("robustness_score", "Robustness score ↑")
]

leader_means = leader_df.groupby(["scenario", "strategy"])[[m for m, _ in plot_metrics]].mean().reset_index()

fig, axes = plt.subplots(2, 2, figsize=(13, 8))
axes = axes.ravel()

for ax, (metric, title) in zip(axes, plot_metrics):
    pivot = leader_means.pivot(index="scenario", columns="strategy", values=metric)
    pivot = pivot.loc[["sparse", "medium", "dense", "corridor"], leader_strategies]
    pivot.plot(kind="bar", ax=ax)
    ax.set_title(title)
    ax.set_xlabel("")
    ax.set_ylabel("Mean value")
    ax.grid(True, axis="y", alpha=0.3)
    ax.legend().remove()

handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc="lower center", ncol=4)
plt.suptitle("Stage 1: Leader Reassignment Validation across Scenarios", fontsize=14)
plt.tight_layout(rect=[0, 0.07, 1, 0.95])

fig_path = FIG_DIR / "fig1_compact_leader_validation.png"
plt.savefig(fig_path, dpi=300)
plt.show()

print("Saved:", fig_path)

# 7. Stage 1b: Parameter Sensitivity and Switching Threshold Analysis

This section selects a reasonable leader configuration for the follower experiment.

The follower experiment will not use arbitrary leader settings. Instead, it uses the perception-aware leader configuration supported by this section.

In [ ]:
# =============================
# 7.1 Parameter Sensitivity
# =============================

selected_scenarios = ["medium", "dense", "corridor"]
wp_values = [0.3, 0.4, 0.5, 0.6, 0.7, 0.8]

sensitivity_records = []

for scenario in selected_scenarios:
    for wp in wp_values:
        remaining = 1 - wp
        wc = remaining * 0.6
        wg = remaining * 0.4

        for trial in range(N_TRIALS_SENSITIVITY):
            row = simulate_leader_trial(
                strategy="perception",
                scenario_name=scenario,
                wp=wp,
                wc=wc,
                wg=wg,
                delta=0.10
            )
            row.update({"trial": trial + 1, "wp": wp, "wc": wc, "wg": wg})
            sensitivity_records.append(row)

sensitivity_df = pd.DataFrame(sensitivity_records)
sensitivity_path = OUTPUT_DIR / "leader_parameter_sensitivity_results.csv"
sensitivity_df.to_csv(sensitivity_path, index=False)

sensitivity_summary = sensitivity_df.groupby(["scenario", "wp"])[leader_metrics].mean().reset_index()
sensitivity_summary_path = TABLE_DIR / "leader_parameter_sensitivity_summary.csv"
sensitivity_summary.to_csv(sensitivity_summary_path, index=False)

print("Saved:", sensitivity_path)
print("Saved:", sensitivity_summary_path)
sensitivity_summary.head()

In [ ]:
# =============================
# 7.2 Switching Threshold Analysis
# =============================

delta_values = [0.00, 0.05, 0.10, 0.15, 0.20, 0.25, 0.30]

switching_records = []

for scenario in selected_scenarios:
    for delta in delta_values:
        for trial in range(N_TRIALS_SENSITIVITY):
            row = simulate_leader_trial(
                strategy="perception",
                scenario_name=scenario,
                wp=0.62,
                wc=0.23,
                wg=0.15,
                delta=delta
            )
            row.update({"trial": trial + 1, "delta": delta})
            switching_records.append(row)

switching_df = pd.DataFrame(switching_records)
switching_path = OUTPUT_DIR / "leader_switching_threshold_results.csv"
switching_df.to_csv(switching_path, index=False)

switching_summary = switching_df.groupby(["scenario", "delta"])[leader_metrics].mean().reset_index()
switching_summary_path = TABLE_DIR / "leader_switching_threshold_summary.csv"
switching_summary.to_csv(switching_summary_path, index=False)

print("Saved:", switching_path)
print("Saved:", switching_summary_path)
switching_summary.head()

In [ ]:
# =============================
# 8. Compact Figure 2: Parameter and Switching Trade-off
# =============================

fig, axes = plt.subplots(2, 2, figsize=(12, 8))

# Aggregate over selected complex scenarios for compact reporting.
sens_agg = sensitivity_summary.groupby("wp")[["robustness_score", "switch_count", "collision_risk", "navigation_success"]].mean().reset_index()
switch_agg = switching_summary.groupby("delta")[["robustness_score", "switch_count", "collision_risk", "navigation_success"]].mean().reset_index()

axes[0, 0].plot(sens_agg["wp"], sens_agg["robustness_score"], marker="o")
axes[0, 0].set_title("Perception weight vs robustness")
axes[0, 0].set_xlabel("Perception weight $w_p$")
axes[0, 0].set_ylabel("Robustness score")
axes[0, 0].grid(True, alpha=0.3)

axes[0, 1].plot(sens_agg["wp"], sens_agg["switch_count"], marker="o")
axes[0, 1].set_title("Perception weight vs switch count")
axes[0, 1].set_xlabel("Perception weight $w_p$")
axes[0, 1].set_ylabel("Switch count")
axes[0, 1].grid(True, alpha=0.3)

axes[1, 0].plot(switch_agg["delta"], switch_agg["robustness_score"], marker="o")
axes[1, 0].set_title("Switching threshold vs robustness")
axes[1, 0].set_xlabel("Switching threshold $\\delta$")
axes[1, 0].set_ylabel("Robustness score")
axes[1, 0].grid(True, alpha=0.3)

axes[1, 1].plot(switch_agg["delta"], switch_agg["switch_count"], marker="o")
axes[1, 1].set_title("Switching threshold vs switch count")
axes[1, 1].set_xlabel("Switching threshold $\\delta$")
axes[1, 1].set_ylabel("Switch count")
axes[1, 1].grid(True, alpha=0.3)

plt.suptitle("Stage 1b: Parameter Sensitivity and Switching Trade-off", fontsize=14)
plt.tight_layout(rect=[0, 0, 1, 0.95])

fig_path = FIG_DIR / "fig2_compact_parameter_switching_tradeoff.png"
plt.savefig(fig_path, dpi=300)
plt.show()

print("Saved:", fig_path)

In [ ]:
# =============================
# 9. Bridge: Select Leader Configuration for Follower Experiment
# =============================

# The previous sensitivity and switching analysis indicates that a moderate-high perception weight
# and a moderate switching threshold give a practical balance between adaptability and stability.
# We use this configuration in Stage 2 rather than selecting follower parameters independently.

leader_selected_config = {
    "leader_strategy": "perception",
    "wp": 0.62,
    "wc": 0.23,
    "wg": 0.15,
    "delta": 0.10,
    "selected_from": "Stage 1 leader baseline, parameter sensitivity, and switching-threshold analysis",
    "used_for": "Stage 2 follower formation response ablation"
}

config_path = CONFIG_DIR / "leader_selected_config_for_follower_stage.json"
with open(config_path, "w", encoding="utf-8") as f:
    json.dump(leader_selected_config, f, indent=2)

print("Saved:", config_path)
leader_selected_config

# 10. Stage 2: Follower Formation Response Ablation

This stage fixes the perception-aware leader reassignment mechanism selected above and evaluates follower-side behaviour after leader switching.

Three follower modes are compared:

1. **Leader-only reassignment**: followers directly track the new leader reference.
2. **Naive reference smoothing**: followers use continuous reference smoothing, which may cause lag.
3. **Proposed adaptive follower response**: followers use transition-limited smoothing, adaptive recovery gain, velocity synchronization, and inter-UAV distance regulation.

This directly addresses the follower-side technical contribution.

In [ ]:
# =============================
# 11. Spatial Follower Simulation Settings
# =============================

with open(config_path, "r", encoding="utf-8") as f:
    leader_config = json.load(f)

spatial_n_uavs = 5
spatial_steps = 140
world_size = 23

start = np.array([2.0, 2.0])
goal = np.array([21.0, 21.0])

formation_offsets = np.array([
    [0.0, 0.0],
    [-1.2, -0.8],
    [-1.2, 0.8],
    [-2.4, -0.8],
    [-2.4, 0.8],
])

scenario_obstacles = {
    "dense": np.array([
        [5.0, 5.0, 1.2],
        [7.0, 7.0, 1.3],
        [9.8, 9.8, 1.4],
        [12.4, 12.0, 1.3],
        [14.8, 14.0, 1.2],
        [8.3, 14.0, 1.2],
        [13.4, 7.8, 1.2],
        [16.0, 10.2, 1.1],
    ]),
    "corridor": np.array([
        [6.0, 4.5, 1.4],
        [6.0, 8.0, 1.4],
        [6.0, 11.5, 1.4],
        [11.0, 7.0, 1.4],
        [11.0, 10.5, 1.4],
        [15.5, 12.0, 1.4],
        [15.5, 15.5, 1.4],
        [10.0, 15.5, 1.2],
    ])
}

follower_modes = ["leader_only", "naive_smoothing", "adaptive_response"]

mode_names = {
    "leader_only": "Leader-only reassignment",
    "naive_smoothing": "Naive reference smoothing",
    "adaptive_response": "Proposed adaptive follower response"
}

print("Follower stage uses leader config:")
leader_config

In [ ]:
# =============================
# 12. Spatial Helper Functions
# =============================

def obstacle_repulsion(pos, obstacles):
    rep = np.zeros(2)
    min_clearance = 10.0

    for ox, oy, r in obstacles:
        obs = np.array([ox, oy])
        diff = pos - obs
        dist = np.linalg.norm(diff) + 1e-6
        clearance = dist - r
        min_clearance = min(min_clearance, clearance)

        if dist < r + 2.9:
            rep += (diff / dist) * (1.0 / max(clearance, 0.15)) * 0.48

    return rep, min_clearance


def perception_score_spatial(pos, obstacles):
    _, clearance = obstacle_repulsion(pos, obstacles)
    score = np.clip(clearance / 4.2 + np.random.normal(0, 0.02), 0, 1)
    return float(score)


def communication_score_spatial(pos, centroid):
    dist = np.linalg.norm(pos - centroid)
    return float(np.clip(1 - dist / 7.0 + np.random.normal(0, 0.015), 0, 1))


def geometry_score_spatial(pos, centroid):
    dist = np.linalg.norm(pos - centroid)
    return float(np.clip(1 - dist / 6.0, 0, 1))


def pairwise_distances(positions):
    dists = []
    for i in range(len(positions)):
        for j in range(i + 1, len(positions)):
            dists.append(np.linalg.norm(positions[i] - positions[j]))
    return np.array(dists)


def distance_regulation_force(i, positions, min_allowed=0.70, max_allowed=4.0):
    force = np.zeros(2)
    pi = positions[i]

    for j in range(len(positions)):
        if i == j:
            continue
        diff = pi - positions[j]
        dist = np.linalg.norm(diff) + 1e-6
        direction = diff / dist

        if dist < min_allowed:
            force += direction * (min_allowed - dist) * 0.35
        elif dist > max_allowed:
            force -= direction * (dist - max_allowed) * 0.10

    return force

In [ ]:
# =============================
# 13. Stage 2 Follower Trial Simulation
# =============================

def simulate_follower_trial(scenario="dense", follower_mode="adaptive_response", leader_config=None):
    if leader_config is None:
        leader_config = {
            "wp": 0.62,
            "wc": 0.23,
            "wg": 0.15,
            "delta": 0.10
        }

    wp = leader_config["wp"]
    wc = leader_config["wc"]
    wg = leader_config["wg"]
    delta = leader_config["delta"]

    obstacles = scenario_obstacles[scenario]

    positions = np.array([start + offset for offset in formation_offsets], dtype=float)
    velocities = np.zeros_like(positions)
    smoothed_refs = positions.copy()

    current_leader = 0
    switch_count = 0
    last_switch_time = -999
    transition_horizon = 10

    trajectories = [[p.copy()] for p in positions]
    leader_history = []
    formation_errors = []
    collision_flags = []
    distance_violation_flags = []
    recovery_times = []
    switch_times = []

    for t in range(spatial_steps):
        centroid = positions.mean(axis=0)

        scores = []
        for p in positions:
            P = perception_score_spatial(p, obstacles)
            C = communication_score_spatial(p, centroid)
            G = geometry_score_spatial(p, centroid)
            scores.append(wp * P + wc * C + wg * G)

        candidate = int(np.argmax(scores))
        if scores[candidate] - scores[current_leader] > delta:
            new_leader = candidate
        else:
            new_leader = current_leader

        if new_leader != current_leader:
            switch_count += 1
            switch_times.append(t)
            last_switch_time = t
        current_leader = new_leader
        leader_history.append(current_leader)

        leader_pos = positions[current_leader]
        goal_vec = goal - leader_pos
        goal_vec = goal_vec / (np.linalg.norm(goal_vec) + 1e-6)
        rep_leader, _ = obstacle_repulsion(leader_pos, obstacles)
        leader_velocity = 0.22 * goal_vec + 0.15 * rep_leader
        leader_new = leader_pos + leader_velocity

        desired_refs = np.zeros_like(positions)
        for i in range(spatial_n_uavs):
            desired_refs[i] = leader_new + (formation_offsets[i] - formation_offsets[current_leader])

        rel = positions - positions[current_leader]
        target_rel = formation_offsets - formation_offsets[current_leader]
        current_formation_error = np.mean(np.linalg.norm(rel - target_rel, axis=1))

        new_positions = positions.copy()
        new_velocities = velocities.copy()

        in_transition = (t - last_switch_time) <= transition_horizon

        for i in range(spatial_n_uavs):
            rep_i, _ = obstacle_repulsion(positions[i], obstacles)

            if i == current_leader:
                new_velocities[i] = leader_velocity
                new_positions[i] = positions[i] + new_velocities[i]
                smoothed_refs[i] = new_positions[i]
                continue

            if follower_mode == "leader_only":
                reference = desired_refs[i]
                kp = 0.25
                kv = 0.05
                dist_force = np.zeros(2)

            elif follower_mode == "naive_smoothing":
                smoothed_refs[i] = 0.82 * smoothed_refs[i] + 0.18 * desired_refs[i]
                reference = smoothed_refs[i]
                kp = 0.17
                kv = 0.08
                dist_force = np.zeros(2)

            elif follower_mode == "adaptive_response":
                # Smoothing is only used around leader-switch transitions.
                if in_transition:
                    smoothed_refs[i] = 0.50 * smoothed_refs[i] + 0.50 * desired_refs[i]
                    reference = smoothed_refs[i]
                else:
                    reference = desired_refs[i]
                    smoothed_refs[i] = desired_refs[i]

                # Increase recovery gain when formation error grows.
                kp = 0.24 + 0.18 * np.clip(current_formation_error, 0, 1.2)
                kv = 0.16
                dist_force = distance_regulation_force(i, positions)

            else:
                raise ValueError("Unknown follower mode")

            tracking = kp * (reference - positions[i])
            synchronization = kv * (leader_velocity - velocities[i])
            obstacle_term = 0.07 * rep_i

            new_velocities[i] = tracking + synchronization + obstacle_term + dist_force
            speed = np.linalg.norm(new_velocities[i])
            if speed > 0.55:
                new_velocities[i] = new_velocities[i] / speed * 0.55

            new_positions[i] = positions[i] + new_velocities[i]

        positions = np.clip(new_positions, 0.5, world_size - 0.5)
        velocities = new_velocities

        for i in range(spatial_n_uavs):
            trajectories[i].append(positions[i].copy())

        rel = positions - positions[current_leader]
        target_rel = formation_offsets - formation_offsets[current_leader]
        formation_error = np.mean(np.linalg.norm(rel - target_rel, axis=1))
        formation_errors.append(formation_error)

        min_clearance = min(obstacle_repulsion(p, obstacles)[1] for p in positions)
        collision_flags.append(1 if min_clearance < 0.35 else 0)

        dists = pairwise_distances(positions)
        violation = np.any(dists < 0.65) or np.any(dists > 4.4)
        distance_violation_flags.append(1 if violation else 0)

    # Recovery time after switches: time to return below a formation threshold.
    recovery_threshold = 0.55
    for st in switch_times:
        found = False
        for tau in range(st, len(formation_errors)):
            if formation_errors[tau] < recovery_threshold:
                recovery_times.append(tau - st)
                found = True
                break
        if not found:
            recovery_times.append(spatial_steps - st)

    mean_recovery_time = float(np.mean(recovery_times)) if recovery_times else 0.0

    trajectories = [np.array(x) for x in trajectories]
    final_positions = np.array([traj[-1] for traj in trajectories])
    final_dist = np.linalg.norm(final_positions.mean(axis=0) - goal)

    navigation_success = float(np.clip(1 - final_dist / 24 - 0.45 * np.mean(collision_flags), 0, 1))
    collision_risk = float(np.mean(collision_flags))
    formation_error_mean = float(np.mean(formation_errors))
    distance_violation_rate = float(np.mean(distance_violation_flags))

    robustness_score = float(
        navigation_success
        - 0.25 * collision_risk
        - 0.18 * formation_error_mean
        - 0.12 * distance_violation_rate
        - 0.02 * switch_count / spatial_steps
    )

    return {
        "scenario": scenario,
        "follower_mode": follower_mode,
        "switch_count": switch_count,
        "mean_recovery_time": mean_recovery_time,
        "collision_risk": collision_risk,
        "formation_error": formation_error_mean,
        "distance_violation_rate": distance_violation_rate,
        "navigation_success": navigation_success,
        "robustness_score": robustness_score,
        "trajectories": trajectories,
        "leader_history": leader_history,
        "formation_errors": formation_errors,
        "switch_times": switch_times
    }

print("Follower-response simulation loaded.")

In [ ]:
# =============================
# 14. Run Stage 2: Follower Response Ablation
# =============================

follower_records = []

for scenario in ["dense", "corridor"]:
    for mode in follower_modes:
        for trial in range(N_TRIALS_FOLLOWER):
            row = simulate_follower_trial(
                scenario=scenario,
                follower_mode=mode,
                leader_config=leader_config
            )
            follower_records.append({
                "scenario": scenario,
                "follower_mode": mode,
                "trial": trial + 1,
                "switch_count": row["switch_count"],
                "mean_recovery_time": row["mean_recovery_time"],
                "collision_risk": row["collision_risk"],
                "formation_error": row["formation_error"],
                "distance_violation_rate": row["distance_violation_rate"],
                "navigation_success": row["navigation_success"],
                "robustness_score": row["robustness_score"]
            })

follower_df = pd.DataFrame(follower_records)
follower_results_path = OUTPUT_DIR / "follower_response_ablation_results.csv"
follower_df.to_csv(follower_results_path, index=False)

print("Saved:", follower_results_path)
follower_df.head()

In [ ]:
# =============================
# 15. Stage 2 Summary and Compact Follower Table
# =============================

follower_metrics = [
    "formation_error",
    "distance_violation_rate",
    "collision_risk",
    "navigation_success",
    "robustness_score",
    "switch_count",
    "mean_recovery_time"
]

follower_summary = follower_df.groupby(["scenario", "follower_mode"])[follower_metrics].agg(["mean", "std"]).reset_index()
follower_summary.columns = ["_".join(c).rstrip("_") for c in follower_summary.columns]

follower_summary_path = TABLE_DIR / "follower_response_ablation_summary.csv"
follower_summary.to_csv(follower_summary_path, index=False)

follower_table = follower_df.groupby(["scenario", "follower_mode"])[[
    "formation_error",
    "distance_violation_rate",
    "collision_risk",
    "navigation_success",
    "robustness_score"
]].mean().reset_index().round(3)

follower_table_path = TABLE_DIR / "paper_table_stage2_follower_ablation.csv"
follower_table.to_csv(follower_table_path, index=False)

print("Saved:", follower_summary_path)
print("Saved:", follower_table_path)
follower_table

In [ ]:
# =============================
# 16. Compact Figure 3: Follower Ablation Metrics
# =============================

compact_metrics = [
    "formation_error",
    "distance_violation_rate",
    "collision_risk",
    "navigation_success",
    "robustness_score"
]

# Dense case is usually the clearest compact result for the main report.
dense_means = follower_df[follower_df["scenario"] == "dense"].groupby("follower_mode")[compact_metrics].mean()
dense_means = dense_means.loc[follower_modes]

plt.figure(figsize=(12, 5))
dense_means.T.plot(kind="bar", figsize=(12, 5))
plt.title("Stage 2: Follower Response Ablation under Dense Environment")
plt.ylabel("Mean metric value")
plt.xlabel("")
plt.xticks(rotation=25, ha="right")
plt.grid(True, axis="y", alpha=0.3)
plt.legend([mode_names[m] for m in follower_modes], loc="best")
plt.tight_layout()

fig_path = FIG_DIR / "fig3_compact_follower_ablation_dense.png"
plt.savefig(fig_path, dpi=300)
plt.show()

print("Saved:", fig_path)

In [ ]:
# =============================
# 17. Compact Figure 4: Formation Error Time Response and Trajectory Comparison
# =============================

example_results = {
    mode: simulate_follower_trial(
        scenario="dense",
        follower_mode=mode,
        leader_config=leader_config
    )
    for mode in follower_modes
}

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: formation error time response.
for mode in follower_modes:
    axes[0].plot(example_results[mode]["formation_errors"], label=mode_names[mode])

for st in example_results["adaptive_response"]["switch_times"]:
    axes[0].axvline(st, linestyle="--", alpha=0.12)

axes[0].set_title("Formation error after leader switches")
axes[0].set_xlabel("Time step")
axes[0].set_ylabel("Formation error")
axes[0].grid(True, alpha=0.3)
axes[0].legend()

# Right: compact dense trajectory for proposed method only.
ax = axes[1]
result = example_results["adaptive_response"]
obstacles = scenario_obstacles["dense"]

for ox, oy, r in obstacles:
    ax.add_patch(Circle((ox, oy), r, alpha=0.25))
    ax.add_patch(Circle((ox, oy), r + 0.6, alpha=0.08))

ax.scatter(start[0], start[1], marker="s", s=80, label="Start")
ax.scatter(goal[0], goal[1], marker="*", s=160, label="Goal")

for i, traj in enumerate(result["trajectories"]):
    ax.plot(traj[:, 0], traj[:, 1], linewidth=1.4, alpha=0.9)
    ax.scatter(traj[-1, 0], traj[-1, 1], s=25)

leader_pts = []
for t, leader_id in enumerate(result["leader_history"]):
    leader_pts.append(result["trajectories"][leader_id][t])
leader_pts = np.array(leader_pts)
ax.plot(leader_pts[:, 0], leader_pts[:, 1], linestyle="--", linewidth=2.2, label="Leader path")

ax.set_xlim(0, world_size)
ax.set_ylim(0, world_size)
ax.set_aspect("equal")
ax.set_title("Proposed response trajectory, dense case")
ax.grid(True, alpha=0.25)
ax.legend(loc="lower right")

plt.suptitle("Stage 2: Follower Formation Response Evidence", fontsize=14)
plt.tight_layout(rect=[0, 0, 1, 0.94])

fig_path = FIG_DIR / "fig4_follower_time_response_and_trajectory.png"
plt.savefig(fig_path, dpi=300)
plt.show()

print("Saved:", fig_path)

In [ ]:
# =============================
# 18. Paper-Level Summary Table
# =============================

# This table is designed for the Results section.
# It combines the most important numerical conclusions into one compact CSV.

# Stage 1: perception-aware gain over best non-perception baseline in complex scenarios.
complex_scenarios = ["dense", "corridor"]
summary_rows = []

for scenario in complex_scenarios:
    scenario_means = leader_df[leader_df["scenario"] == scenario].groupby("strategy")[leader_metrics].mean()
    perception = scenario_means.loc["perception"]
    baselines = scenario_means.loc[["static", "task", "communication"]]

    best_baseline_robust = baselines["robustness_score"].max()
    best_baseline_success = baselines["navigation_success"].max()
    lowest_baseline_collision = baselines["collision_risk"].min()

    summary_rows.append({
        "stage": "leader_reassignment",
        "scenario": scenario,
        "main_comparison": "perception vs best baseline",
        "collision_risk_change": round(perception["collision_risk"] - lowest_baseline_collision, 3),
        "navigation_success_gain": round(perception["navigation_success"] - best_baseline_success, 3),
        "robustness_gain": round(perception["robustness_score"] - best_baseline_robust, 3),
        "interpretation": "Perception-aware reassignment improves complex-environment robustness"
    })

# Stage 2: adaptive follower response gain over leader-only.
for scenario in complex_scenarios:
    scenario_means = follower_df[follower_df["scenario"] == scenario].groupby("follower_mode")[follower_metrics].mean()
    adaptive = scenario_means.loc["adaptive_response"]
    leader_only = scenario_means.loc["leader_only"]

    summary_rows.append({
        "stage": "follower_response",
        "scenario": scenario,
        "main_comparison": "adaptive response vs leader-only",
        "formation_error_change": round(adaptive["formation_error"] - leader_only["formation_error"], 3),
        "distance_violation_change": round(adaptive["distance_violation_rate"] - leader_only["distance_violation_rate"], 3),
        "navigation_success_gain": round(adaptive["navigation_success"] - leader_only["navigation_success"], 3),
        "robustness_gain": round(adaptive["robustness_score"] - leader_only["robustness_score"], 3),
        "interpretation": "Adaptive follower response preserves formation after leader switching"
    })

paper_summary_table = pd.DataFrame(summary_rows)
paper_summary_path = TABLE_DIR / "paper_level_key_results_summary.csv"
paper_summary_table.to_csv(paper_summary_path, index=False)

print("Saved:", paper_summary_path)
paper_summary_table